In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
=============================================================================
 DNA VIRAL GENOME TRANSCRIPTIONAL-ACTIVITY CLASSIFICATION
=============================================================================
Classifies DNA viral genomes into three transcriptional-activity categories
from a gene x sample expression (adjusted coverage) table, and produces the
summary tables and figures used in the manuscript.

--------------------------------------------------------------------------
CATEGORIES (per genome; coverage threshold = 1x)
--------------------------------------------------------------------------
  inactive        : no gene expressed above the coverage threshold.
  active-unknown  : at least one gene expressed above threshold, but NONE of
                    them is a structural, lysis, or lysogenic marker
                    (i.e. expression limited to replication/regulatory/other
                    genes). Merges the former 'non-lytic-active' and
                    'lysogenic-active' classes.
  lytic-active    : at least one structural or lysis gene expressed above
                    threshold (capsid, tail, portal, terminase, packaging,
                    holin, endolysin, ...).
                    
Rationale: this scheme summarises the ACTIVE CYCLE status of each genome
(is it expressing productive-cycle / structural-lysis genes or not), and is
independent of the temperate-vs-virulent LIFESTYLE classification, which is
based on gene content (presence of lysogeny genes) and reported separately.

NOTE on structural genes: expression of structural genes (e.g. capsid) is
used here as a signature of productive/lytic-cycle transcription. It
indicates active viral gene expression, not confirmed cell lysis.

--------------------------------------------------------------------------
TWO LEVELS OF AGGREGATION
--------------------------------------------------------------------------
  (1) PER-SAMPLE  : each genome is classified within each individual sample;
                    proportions are computed per sample. This is the basis of
                    the STATISTICS (Kruskal-Wallis across treatments), because
                    samples are independent replicates and unequal sample
                    sizes per treatment are handled correctly. Used in the
                    manuscript.

  (2) PER-TREATMENT (pooled): a genome is 'active' in a treatment if it is
                    active in ANY sample of that treatment. Because treatments
                    can have different numbers of samples, this pooling can
                    inflate the 'active' fraction in treatments with more
                    samples; it is provided for completeness only.

--------------------------------------------------------------------------
OUTPUTS  (written to OUTDIR)
--------------------------------------------------------------------------
  Tables (CSV):
    activity_per_sample.csv               per-sample % in each category
    activity_percent_per_sample_mean.csv  per-treatment mean of per-sample %
    activity_counts_per_treatment.csv     pooled counts (per-treatment)
    activity_percent_per_treatment.csv    pooled % (per-treatment)
    activity_kruskal_stats.csv            Kruskal-Wallis + BH-FDR per category
    genome_classification_per_sample.csv  per genome x sample category (long)
  Figures (PDF + SVG):
    Fig4E_stacked_persample               stacked mean +/- SEM (per-sample)
    Fig4E_boxplots_persample              grouped boxplots (per-sample)
    Fig4E_stacked_per_treatment           pooled stacked bar (per-treatment)

--------------------------------------------------------------------------
USAGE
--------------------------------------------------------------------------
  1. Edit CONFIG below (INPUT path, SAMPLES, OUTDIR, THRESHOLD if needed).
  2. Run:  python viral_activity_classification.py
  Requires: pandas, numpy, scipy, matplotlib.
=============================================================================
"""

"\n=============================================================================\n DNA VIRAL GENOME TRANSCRIPTIONAL-ACTIVITY CLASSIFICATION\n=============================================================================\nClassifies DNA viral genomes into three transcriptional-activity categories\nfrom a gene x sample expression (adjusted coverage) table, and produces the\nsummary tables and figures used in the manuscript.\n\n--------------------------------------------------------------------------\nCATEGORIES (per genome; coverage threshold = 1x)\n--------------------------------------------------------------------------\n  inactive        : no gene expressed above the coverage threshold.\n  active-unknown  : at least one gene expressed above threshold, but NONE of\n                    them is a structural, lysis, or lysogenic marker\n                    (i.e. expression limited to replication/regulatory/other\n                    genes). Merges the former 'non-lytic-active' and\n     

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.stats import kruskal, false_discovery_control

In [10]:
# =================
# CONFIG
# =================
INPUT = "data/Table_Cover_genes_Activegenomes.txt"  # gene x sample table (tab-sep)
OUTDIR = "./activity_outputs"                # output folder
COVERAGE_THRESHOLD = 1                       # a gene is "expressed" if coverage > this

# Samples to include, grouped by treatment.
TREATMENTS = {
    "Unmanip": ["Unmanip_1","Unmanip_2","Unmanip_3","Unmanip_4","Unmanip_5","Unmanip_6","Unmanip_7","Unmanip_8"],
    "HMM":      ["HMM_1","HMM_2","HMM_3","HMM_4","HMM_5","HMM_6","HMM_7","HMM_8"],
    "HM":       ["HM_1","HM_2","HM_3","HM_4","HM_5","HM_6","HM_7","HM_8"],
    "LM":       ["LM_2","LM_3","LM_4","LM_5","LM_6","LM_7","LM_8"],
}
# Display order for treatments in tables and figures.
TREATMENT_ORDER = ["Unmanip", "HMM", "HM", "LM"]

# Column names in the input identifying the genome and gene.
GENOME_COL = "Genome"
GENE_COL   = "Gene"

# Colours per category (used in all figures).
COLORS = {
    "inactive":       "#B0B0B0",
    "active-unknown": "#457B9D",
    "lytic-active":   "#E76F51",
}
CATEGORY_ORDER = ["inactive", "active-unknown", "lytic-active"]


In [11]:
# =========================================================================
# MARKER KEYWORD SETS
#   structural  -> capsid/tail/portal/terminase/packaging/DNA_polymerase...
#   lytic_kw    -> structural + lysis genes (holin/endolysin/lysozyme...)
#   lysogenic   -> integration/excision machinery (integrase/recombinase...)
# Matching is case-insensitive substring matching on the gene name.
# =========================================================================
STRUCTURAL = [
    "head-tail_joining", "head-tail_adaptor", "DNA_polymerase", "tail", "head",
    "packaging", "capsid", "lytic", "terminase", "portal", "structural",
]
LYTIC_KW = STRUCTURAL + ["holin", "endolysin", "lysozyme", "exonuclease"]
LYSOGENIC_KW = [
    "site-specific recombinase", "integrase", "integration", "recombinase",
    "xerc", "xerd", "prophage", "excisionase", "excision",
    "attachment site", "attp", "attb", "integron", "lysogen", "lysogenic",
]



In [12]:
# =========================================================================
# HELPERS
# =========================================================================
def has_keyword(gene_name, keywords):
    """Case-insensitive substring match of any keyword in a gene name."""
    g = gene_name.lower()
    return any(k.lower() in g for k in keywords)


def classify_block(expr_block, gene_names, threshold):
    """
    Classify one genome from an expression block (genes x samples).
    Returns one of: 'inactive', 'active-unknown', 'lytic-active'.
    A genome is 'lytic-active' if any structural/lysis gene is expressed
    (>threshold) in any of the block's samples; else 'active-unknown' if any
    gene is expressed; else 'inactive'.
    """
    is_active = (expr_block > threshold).values.any()
    is_lytic = any(
        has_keyword(g, LYTIC_KW) and (expr_block.iloc[i] > threshold).any()
        for i, g in enumerate(gene_names)
    )
    if is_lytic:
        return "lytic-active"
    if is_active:
        return "active-unknown"
    return "inactive"
    
def load_table(path, sample_cols):
    """Load the gene x sample table and coerce sample columns to numeric."""
    df = pd.read_csv(path, sep="\t", quoting=3, dtype=str, keep_default_na=False)
    df.columns = df.columns.str.strip()
    missing = [c for c in sample_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Sample columns not found in input: {missing}")
    for c in sample_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
    return df

In [13]:
# =========================================================================
# CLASSIFICATION
# =========================================================================

def classify_per_sample(df, treatments, threshold):
    """One category per (genome, sample); returns a tidy DataFrame of
    per-sample category percentages plus a long genome-level table."""
    rows, long_rows = [], []
    grouped = list(df.groupby(GENOME_COL))
    for cond, samples in treatments.items():
        for s in samples:
            cats = {}
            for genome, grp in grouped:
                cat = classify_block(grp[[s]], grp[GENE_COL].tolist(), threshold)
                cats[genome] = cat
                long_rows.append({"treatment": cond, "sample": s,
                                  "genome": genome, "category": cat})
            vc = pd.Series(cats).value_counts()
            total = vc.sum()
            rows.append({"sample": s, "treatment": cond,
                         **{c: 100 * vc.get(c, 0) / total for c in CATEGORY_ORDER}})
    per_sample = pd.DataFrame(rows)
    long_df = pd.DataFrame(long_rows)
    return per_sample, long_df


def classify_per_treatment(df, treatments, threshold):
    """One category per (genome, treatment): active if active in ANY sample of
    the treatment. Returns pooled counts and percentages."""
    counts = {}
    grouped = list(df.groupby(GENOME_COL))
    for cond, samples in treatments.items():
        cats = {genome: classify_block(grp[samples], grp[GENE_COL].tolist(), threshold)
                for genome, grp in grouped}
        counts[cond] = pd.Series(cats).value_counts()
    summary = pd.DataFrame(counts).fillna(0).astype(int).T
    summary = summary[[c for c in CATEGORY_ORDER if c in summary.columns]]
    summary = summary.loc[[t for t in TREATMENT_ORDER if t in summary.index]]
    percent = summary.div(summary.sum(axis=1), axis=0) * 100
    return summary, percent


def kruskal_stats(per_sample, order):
    """Kruskal-Wallis across treatments for each category (per-sample %),
    with Benjamini-Hochberg FDR correction."""
    recs, pvals = [], []
    for cat in CATEGORY_ORDER:
        groups = [per_sample.loc[per_sample.treatment == t, cat].values for t in order]
        H, p = kruskal(*groups)
        recs.append({"category": cat, "kruskal_H": H, "kruskal_p": p})
        pvals.append(p)
    padj = false_discovery_control(pvals)  # Benjamini-Hochberg
    stats = pd.DataFrame(recs)
    stats["kruskal_p_BH"] = padj
    return stats

In [14]:
# =========================================================================
# FIGURES
# =========================================================================

def _save(fig, stem, outdir):
    """Save a figure as both PDF and SVG."""
    fig.savefig(os.path.join(outdir, stem + ".pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(outdir, stem + ".svg"), bbox_inches="tight")
    plt.close(fig)


def fig_stacked_persample(per_sample, order, outdir, stem="Fig4E_stacked_persample"):
    """Stacked bar of per-sample MEAN percentages (+/- SEM) per treatment."""
    means = per_sample.groupby("treatment")[CATEGORY_ORDER].mean().loc[order]
    sems  = per_sample.groupby("treatment")[CATEGORY_ORDER].sem().loc[order]
    fig, ax = plt.subplots(figsize=(7, 5))
    bottom = np.zeros(len(order))
    for cat in CATEGORY_ORDER:
        ax.bar(order, means[cat], bottom=bottom, label=cat,
               color=COLORS[cat], edgecolor="black", width=0.65)
        ax.errorbar(order, bottom + means[cat].values, yerr=sems[cat].values,
                    fmt="none", ecolor="black", capsize=3, lw=1)
        bottom += means[cat].values
    ax.set_ylabel("% of genomes per sample (mean \u00B1 SEM)", fontsize=12)
    ax.set_title("Transcriptional activity of DNA viral genomes", fontsize=12)
    ax.set_ylim(0, 100)
    ax.legend(loc="upper right", fontsize=9, frameon=False)
    fig.tight_layout()
    _save(fig, stem, outdir)


def fig_boxplots_persample(per_sample, stats, order, outdir, stem="Fig4E_boxplots_persample"):
    """Grouped boxplots: one box per category per treatment, with sample points."""
    fig, ax = plt.subplots(figsize=(9, 5))
    w = 0.25
    for j, cat in enumerate(CATEGORY_ORDER):
        positions = [i + (j - 1) * w for i in range(len(order))]
        data = [per_sample.loc[per_sample.treatment == t, cat].values for t in order]
        bp = ax.boxplot(data, positions=positions, widths=w * 0.9, patch_artist=True)
        for patch in bp["boxes"]:
            patch.set_facecolor(COLORS[cat]); patch.set_alpha(0.75)
        for med in bp["medians"]:
            med.set_color("black")
        for i, t in enumerate(order):
            y = per_sample.loc[per_sample.treatment == t, cat].values
            ax.scatter(np.random.normal(positions[i], 0.03, len(y)), y,
                       s=10, color="black", zorder=3)
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order)
    ax.set_ylabel("% of genomes per sample", fontsize=12)
    ax.legend(handles=[Patch(facecolor=COLORS[c], alpha=0.75, label=c) for c in CATEGORY_ORDER],
              loc="upper right", fontsize=9, frameon=False)
    pstr = ", ".join(f"{r.category} p={r.kruskal_p:.2f}" for r in stats.itertuples())
    ax.set_title("Per-sample activity  |  Kruskal-Wallis: " + pstr, fontsize=10)
    fig.tight_layout()
    _save(fig, stem, outdir)


def fig_stacked_per_treatment(percent, order, outdir, stem="Fig4E_stacked_per_treatment"):
    """Pooled per-treatment stacked bar (for completeness / supplement)."""
    percent = percent.loc[[t for t in order if t in percent.index]]
    fig, ax = plt.subplots(figsize=(7, 5))
    bottom = np.zeros(len(percent))
    for cat in CATEGORY_ORDER:
        ax.bar(percent.index, percent[cat], bottom=bottom, label=cat,
               color=COLORS[cat], edgecolor="black", width=0.65)
        bottom += percent[cat].values
    ax.set_ylabel("% of genomes (pooled per treatment)", fontsize=12)
    ax.set_title("DNA viral activity categories (per-treatment, pooled)", fontsize=12)
    ax.set_ylim(0, 100)
    ax.legend(loc="upper right", fontsize=9, frameon=False)
    fig.tight_layout()
    _save(fig, stem, outdir)


In [16]:
# =========================================================================
# MAIN
# =========================================================================

def main():
    os.makedirs(OUTDIR, exist_ok=True)
    all_samples = [s for v in TREATMENTS.values() for s in v]

    print(f"Loading {INPUT} ...")
    df = load_table(INPUT, all_samples)
    n_genomes = df[GENOME_COL].nunique()
    print(f"  {n_genomes} genomes, {len(all_samples)} samples "
          f"({', '.join(f'{k}:{len(v)}' for k, v in TREATMENTS.items())})")

    # --- per-sample classification + stats ---
    per_sample, long_df = classify_per_sample(df, TREATMENTS, COVERAGE_THRESHOLD)
    stats = kruskal_stats(per_sample, TREATMENT_ORDER)
    mean_ps = per_sample.groupby("treatment")[CATEGORY_ORDER].mean().loc[TREATMENT_ORDER]

    # --- per-treatment (pooled) classification ---
    counts, percent = classify_per_treatment(df, TREATMENTS, COVERAGE_THRESHOLD)

    # --- save tables ---
    per_sample.to_csv(os.path.join(OUTDIR, "activity_per_sample.csv"), index=False)
    mean_ps.round(2).to_csv(os.path.join(OUTDIR, "activity_percent_per_sample_mean.csv"))
    counts.to_csv(os.path.join(OUTDIR, "activity_counts_per_treatment.csv"))
    percent.round(2).to_csv(os.path.join(OUTDIR, "activity_percent_per_treatment.csv"))
    stats.round(4).to_csv(os.path.join(OUTDIR, "activity_kruskal_stats.csv"), index=False)
    long_df.to_csv(os.path.join(OUTDIR, "genome_classification_per_sample.csv"), index=False)

    # --- figures (PDF + SVG) ---
    fig_stacked_persample(per_sample, TREATMENT_ORDER, OUTDIR)
    fig_boxplots_persample(per_sample, stats, TREATMENT_ORDER, OUTDIR)
    fig_stacked_per_treatment(percent, TREATMENT_ORDER, OUTDIR)

    # --- console summary ---
    print("\nPER-SAMPLE mean % (basis of statistics):")
    print(mean_ps.round(1).to_string())
    print("\nPER-TREATMENT pooled %:")
    print(percent.round(1).to_string())
    print("\nKRUSKAL-WALLIS across treatments (per-sample):")
    print(stats.round(4).to_string(index=False))

if __name__ == "__main__":
    main()

Loading data/Table_Cover_genes_Activegenomes.txt ...
  141 genomes, 31 samples (Unmanip:8, HMM:8, HM:8, LM:7)

PER-SAMPLE mean % (basis of statistics):
           inactive  active-unknown  lytic-active
treatment                                        
Unmanip        78.6            20.7           0.7
HMM            74.7            23.4           1.9
HM             78.9            19.9           1.2
LM             76.0            22.7           1.3

PER-TREATMENT pooled %:
         inactive  active-unknown  lytic-active
Unmanip      48.2            48.9           2.8
HMM          43.3            49.6           7.1
HM           50.4            45.4           4.3
LM           46.1            50.4           3.5

KRUSKAL-WALLIS across treatments (per-sample):
      category  kruskal_H  kruskal_p  kruskal_p_BH
      inactive     6.7224     0.0813        0.1219
active-unknown     3.8660     0.2763        0.2763
  lytic-active     7.5125     0.0572        0.1219
